# 02 - Preprocessing & Class Imbalance (Zero Data Leakage)
Demonstrates: patient-level split, fit-on-train-only, SMOTE inside the Pipeline.
Full logic: `src/preprocessing.py`.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fossbk-spec/stroke-prediction/blob/main/notebooks/02_preprocessing_and_imbalance.ipynb)
[![Open In Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/fossbk-spec/stroke-prediction/blob/main/notebooks/02_preprocessing_and_imbalance.ipynb)


In [1]:
# Runs offline on Google Colab / Kaggle Notebooks: no internet/Kaggle token needed.
# The dataset CSV is already bundled in the repo at data/raw/.
import os, sys, subprocess
if "google.colab" in sys.modules or os.path.exists("/kaggle/working"):
    if not os.path.exists("stroke-prediction"):
        subprocess.run(["git", "clone", "--depth", "1",
                         "https://github.com/fossbk-spec/stroke-prediction.git"], check=True)
    os.chdir("stroke-prediction")
    subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"], check=False)
sys.path.insert(0, os.getcwd())


In [2]:
from src.data_loader import load_raw_data, split_features_target
from src.preprocessing import patient_level_split, make_full_pipeline, assert_no_fit_on_holdout
from src.models import get_model

df = load_raw_data()
split = patient_level_split(df)
split.summary()

[OK] ZERO DATA LEAKAGE confirmed (Train=3,270 IDs | Validation=818 IDs | Test=1,022 IDs).


,split,n_samples,n_patients,n_positive,positive_rate
0,train,3270,3270,159,0.048624
1,val,818,818,40,0.048900
2,test,1022,1022,50,0.048924


## Verify Zero Data Leakage (assertion code)

In [3]:
from src.preprocessing import verify_zero_data_leakage
verify_zero_data_leakage(split.train, split.val, split.test)

[OK] ZERO DATA LEAKAGE confirmed (Train=3,270 IDs | Validation=818 IDs | Test=1,022 IDs).


True

## Fit the pipeline (Imputer + Scaler + Encoder + SMOTE) on Train ONLY

In [4]:
X_train, y_train = split_features_target(split.train)
X_test, y_test = split_features_target(split.test)

pipeline = make_full_pipeline(get_model('logistic_regression'))
pipeline.fit(X_train, y_train)
assert_no_fit_on_holdout(pipeline, X_test)  # must pass

True

## Save the transformed feature matrix to data/processed/

In [5]:
import pandas as pd
pre = pipeline.named_steps['preprocessor']
processed = pd.DataFrame(pre.transform(X_train), columns=pre.get_feature_names_out())
processed['stroke'] = y_train.to_numpy()
processed.to_csv('data/processed/train_transformed.csv', index=False)
processed.head()

,age,avg_glucose_level,bmi,hypertension,heart_disease,ever_married,gender_Female,gender_Male,gender_Other,work_type_Govt_job,work_type_Never_worked,work_type_Private,work_type_Self-employed,work_type_children,Residence_type_Rural,Residence_type_Urban,smoking_status_Unknown,smoking_status_formerly smoked,smoking_status_never smoked,smoking_status_smokes,stroke
0,1.060995,2.718416,1.008684,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1
1,0.264609,1.446517,0.722018,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1
2,1.370700,-0.792255,-0.190104,1.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1
3,1.149482,-0.254366,-0.789498,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1
4,0.707046,-0.658115,-0.325428,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1
